# PyTorch

## Importar librerías

* `PyTorch` es una biblioteca de `deep learning` desarrollada por `Facebook (Meta)` que permite:
  - Crear redes neuronales.
  - Entrenarlas.
  - Usar GPU fácilmente.
  - Hacer investigación y producción
* Es muy popular porque es:
  - Flexible.
  - Intuitiva (muy estilo Python).
  - Dinámica (gráfica computacional dinámica).
* Los `tensores` son la base de todo.
* Un tensor es como:
  - Un número (0D).
  - Un vector (1D).
  - Una matriz (2D).
  - Un arreglo multidimensional (3D, 4D...).
```python
import torch

x = torch.tensor([1.0, 2.0, 3.0])
print(x)
```

* ***Un tensor es como un numpy array, pero puede: Usar GPU y Calcular gradientes automáticamente***.
* `PyTorch` puede calcular derivadas automáticamente (backpropagation).
* Esto se llama `Autograd (Derivación automática)`.
* Esto es clave para entrenar redes neuronales.
```python
x = torch.tensor(2.0, requires_grad=True)

y = x**2
y.backward()

print(x.grad)  # Derivada de x² en x=2 → 4
```

* `torch.nn as nn` este módulo contiene:
  - Capas (Linear, Conv2d, etc.).
  - Funciones de pérdida.
  - Modelos base.
```python
layer = nn.Linear(10, 5)  # 10 entradas, 5 salidas
```

* `import torch.nn.functional as F` aquí están funciones como:
  - ReLU.
  - Softmax.
  - CrossEntropy.
  - MaxPool.
* ***Estructura típica de una red neuronal***
```python
class MiRed(nn.Module):
    def __init__(self):
        super(MiRed, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
```


In [1]:
import numpy as np
import matplotlib.pyplot as plt

### PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

## Importar imagenes

In [2]:
import gzip
import os
from os.path import isfile, join
from contextlib import contextmanager
import zipfile

### Tener cuidado con el formato de windows
mnist_path = "C:\\Users\\PC\\OneDrive\\Documentos\\MNIST"

def getListFiles(mnist_path : str) -> list:
    return [
        join(mnist_path, file) 
        for file in os.listdir(mnist_path) 
        if isfile(join(mnist_path, file))
    ]


@contextmanager
def open_mnist_file(path: str):
    """
    Abre archivos MNIST en formato:
    - .gz
    - .zip
    - sin comprimir (.idx*-ubyte)
    Devuelve un stream binario.
    """
    if path.endswith(".gz"):
        f = gzip.open(path, "rb")
        try:
            yield f
        finally:
            f.close()

    elif path.endswith(".zip"):
        zf = zipfile.ZipFile(path)
        try:
            inner_name = zf.namelist()[0]
            with zf.open(inner_name) as f:
                yield f
        finally:
            zf.close()

    else:
        with open(path, "rb") as f:
            yield f


def _read_int32_be(f) -> int:
    """Lee un entero de 32 bits big-endian."""
    return int.from_bytes(f.read(4), byteorder="big")


def _read_images(f):
    """Lee un archivo idx3-ubyte (imágenes)."""
    _ = _read_int32_be(f)          # magic number
    num_images = _read_int32_be(f)
    rows = _read_int32_be(f)
    cols = _read_int32_be(f)

    data = f.read()
    images = np.frombuffer(data, dtype=np.uint8)
    return images.reshape(num_images, rows, cols)


def _read_labels(f):
    """Lee un archivo idx1-ubyte (labels)."""
    _ = _read_int32_be(f)  # magic number
    _ = _read_int32_be(f)  # número de labels
    return np.frombuffer(f.read(), dtype=np.uint8)


def getImages(files: list[str]):
    x_train = y_train = x_test = y_test = None

    for path in files:
        with open_mnist_file(path) as f:

            if "train-images" in path:
                x_train = _read_images(f)

            elif "train-labels" in path:
                y_train = _read_labels(f)

            elif "t10k-images" in path:
                x_test = _read_images(f)

            elif "t10k-labels" in path:
                y_test = _read_labels(f)

    return x_train, y_train, x_test, y_test

files = getListFiles(mnist_path)

In [3]:
x_train_num, y_train_num, x_test_num, y_test_num = getImages(files)

### Datos de entrenamiento
x_train = x_train_num[:50000].reshape(50000, -1).astype(np.float32)
y_train = y_train_num[:50000].reshape(50000, 1)

### Datos de validación (los ultimos 10k)
x_val = x_train_num[50000:].reshape(10000, -1).astype(np.float32)
y_val = y_train_num[50000:].reshape(10000, 1)

### Datos de prueba (origialmente ya son 10k, solo se pasaron a vector)
x_test = x_test_num.copy().reshape(10000, -1).astype(np.float32)
y_test = y_test_num.copy().reshape(10000, 1)

## Normalizar Imágenes

In [4]:
### Normalizar para tener una media de 0 y una desv de 1.
def normalise(x_mean, x_std, x_data):
    return (x_data-x_mean)/x_std

In [5]:
x_mean = x_train.mean()
x_std = x_train.std()

x_train = normalise(x_mean, x_std, x_train)
x_val = normalise(x_mean, x_std, x_val)
x_test = normalise(x_mean, x_std, x_test)

print(x_train.mean(), x_train.std())

-3.1638146e-07 0.99999934


## Crear mini batches

In [21]:

def create_minibatches(x, y, mb_size, shuffle = True):
    '''
    x  #muestras, 784
    y #muestras, 1
    '''
    assert x.shape[0] == y.shape[0], 'Error en cantidad de muestras'
    total_data = x.shape[0]
    if shuffle: 
        idxs = np.arange(total_data)
        np.random.shuffle(idxs)
        x = x[idxs]
        y = y[idxs]  
    return ((x[i:i+mb_size], y[i:i+mb_size]) for i in range(0, total_data, mb_size))

## Convertir Numpy array a PyTorch tensor

* Está convirtiendo los datos (probablemente numpy arrays) en tensores de PyTorch.
* ***¿Por qué necesitamos convertirlos?*** PyTorch solo puede:
  - Calcular gradientes.
  - Usar GPU.
  - Pasarlos al modelo.
* Solo si son `torch.Tensor`.
* El modelo espera tensores, no numpy arrays.
* Cuando hacemos `torch.tensor(x_train)` PyTorch crea una copia interna, aquí evitaría un warning.

* `torch.from_numpy(x_train.copy())` es más rápido y evita copias innecesarias
* `x_train_tensor = torch.from_numpy(x_train).float()`

In [6]:
x_train_tensor = torch.tensor(x_train.copy())
y_train_tensor = torch.tensor(y_train.copy())

x_test_tensor = torch.tensor(x_test.copy())
y_test_tensor = torch.tensor(y_test.copy())

x_val_tensor = torch.tensor(x_val.copy())
y_val_tensor = torch.tensor(y_val.copy())

## Usar GPU de estar disponible

* Este código decide dónde se harán los cálculos:
  - `CPU` (procesador normal).
  - `GPU` (tarjeta gráfica).
* ***¿Por qué es importante?***
* Entrenar redes neuronales implica:
  - Multiplicaciones de matrices gigantes.
  - Millones o miles de millones de operaciones.
* Las GPUs están diseñadas para:
  - Procesamiento paralelo masivo.
  - Operaciones matriciales rápidas.
| Dispositivo | Velocidad            |
| ----------- | -------------------- |
| CPU         | Lento                |
| GPU         | MUCHÍSIMO más rápido |

* `torch.cuda.is_available()` --> ***¿Hay una GPU compatible con CUDA disponible?***.
* `torch.device('cuda')`: Crea un objeto que representa el dispositivo GPU.
* `torch.device('cpu')`: Procesador normal.
* Esto solo define el dispositivo, pero no mueve nada automáticamente.
* Hay que mover el modelo `model = model.to(device)` y los datos `data = data.to(device)`.
* Si no haces esto, te saldrá un error típico: `Expected all tensors to be on the same device`.
* Si yo si tengo GPU, pero otro no, no se puede dejar así por eso esta validación.

* ***Concepto profundo***
* En PyTorch, los tensores viven en un dispositivo específico.
```python
x = torch.tensor([1,2,3])
print(x.device) ### Sería cpu

## Lo movemos
x = x.to('cuda')
print(x.device)
```

In [7]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Estamos usando: {device}')

Estamos usando: cpu


## Compute Accuracy

* Aquí estamos calculando `accuracy (exactitud) del modelo`.
$$ \text{accuracy} = \frac{\text{predicciones correctas}}{\text{total de muestras}}$$
* `model` es nuestra red neuronal entrenada.
* `x` datos, `y` etiquetas reales y `mb_size` tamaño del minibatch.
* Al principios inicializamos los valores con ceros.
* `model.eval()` esto poe el modelo en ***modo evaluación***.
* Desactiva cosas como:
  - `Dropout`.
  - `BatchNorm en modo entrenamiento`.
* ***Si no haces esto, la accuracy puede ser incorrecta***.
* `model = model.to(device=device)` Asegura que el modelo esté en el mismo dispositivo que los datos.
* `with torch.no_grad():` esto le dice a PyTorch que no guarde gradientes.
* ***¿Para qué?***
  - Ahorra memoria.
  - Hace el cálculo más rápido.
  - No necesitamos backpropagation.
  - Porque solo estamos evaluando.

* ***¿Por qué long para etiquetas?***
* `nn.CrossEntropyLoss()` exige:
  - Predicciones → `float`.
  - Labels → `long (enteros)`.
* Si no haces esto puede darte error después.
* `scores = model(xi)` esto correspondería al `FowardPass`.
* Ahí el modelo devuelve algo como: `(batch_size, num_clases)`
* Si es NMIST → probablemente 10 clases.
* `_, pred = scores.max(dim=1)` obtener las predicciones.
* `scores.max(dim=1)` devuelve:
  - Valor máximo.
  - Índice del máximo.
* Por ejemplo esto podría regresar `[2.3, -1.2, 0.5] → índice 0`, donde la mayor probabilidad es la primera.
* Nos interesa el índice (la clase predicha).
* Por eso usamos _ para ignorar el valor.
* ***¿Por qué .squeeze()?***
  - Probablemente tus etiquetas tienen forma: `(mb_size, 1)`
  - Y pred tiene forma `(mb_size)`.
* Esto para sacar las clases quesi fueron clasificadas correctamente `pred == yi.squeeze()`.

In [26]:
def accuracy(model, x, y, mb_size):
    num_correct = 0
    num_total = 0
    model.eval()
    model = model.to(device=device)
    with torch.no_grad():
        for (xi, yi) in create_minibatches(x, y, mb_size):
            xi = xi.to(device=device, dtype=torch.float32)
            yi = yi.to(device=device, dtype=torch.long)
            
            scores = model(xi) # mb_size, 10
            
            _, pred = scores.max(dim=1) #pred shape(mb_size)
            
            num_correct += (pred == yi.squeeze()).sum() # pred shape (mb_size), yi shape (mb_size, 1)
            num_total += pred.size(0)
        return float(num_correct)/num_total
            

## Loop de Entrenamiento

* Entrena el modelo durante cierto número de épocas usando mini-batches.
* Recibe:
  - `model` → red neuronal.
  - `optimiser` → Adam, SGD, etc.
  - `mb_size` → tamaño de minibatch.
  - `epochs` → cuántas veces recorrer todo el dataset.

* Una época = pasar todo el dataset una vez.
* Si tienes 10,000 imágenes y mb_size = 100: → 100 mini-batches por época.
* 100 veces un mini-batch.
* `model.train()`: Activa el modo entrenamiento.
* Activa Dropout y Activa BatchNorm en modo actualización.
* ***Nota: normalmente se coloca una vez antes del loop de minibatches, no dentro***.
* Funciona igual, pero es redundante repetirlo en cada batch.
* Aquí también hacemos `FowardPass` `scores = model(xi)`.
* ***Son logits, no probabilidades***.
* ***Muy importante entender esto***
* F.cross_entropy combina internamente:
  - `log_softmax`.
  - `nll_loss`.
* ***Por eso NO debes aplicar softmax antes***.
* `optimiser.zero_grad()` limpiar gradientes.
* En PyTorch los gradientes se acumulan.
* ***Si no limpias, estarías sumando gradientes anteriores, esto es clave***.
* Literal el `Back` `cost.backward()`.
* PyTorch:
  - Calcula derivadas automáticamente.
  - Usa autograd.
  - Llena `.grad` en cada parámetro.
* `optimiser.step()` actualización de pesos.
* Aquí el optimizador:
  - Lee los gradientes.
  - Actualiza los pesos.
* Esto parece que es el `Stochatist Decedent Gradient`. W menos el learning rate por dL/dW.

In [24]:
def train(model, optimiser, mb_size, epochs=100):
    ### Movemos a donde esta el modelo.
    model = model.to(device=device)
    for epoch in range(epochs):
        for (xi, yi) in create_minibatches(x_train_tensor, y_train_tensor, mb_size):
            model.train()
            xi = xi.to(device=device, dtype=torch.float32)
            yi = yi.to(device=device, dtype=torch.long)
            scores = model(xi)
            # funcion cost
            cost = F.cross_entropy(input= scores, target=yi.squeeze())
            optimiser.zero_grad()
            cost.backward()
            optimiser.step()
            
        print(f'Epoch: {epoch}, costo: {cost.item()}, accuracy: {accuracy(model, x_val_tensor, y_val_tensor, mb_size)}')
    
            

## Modelo usando Sequencial

* ***¿Por qué son logits y no predicciones? ¿Qué es un logit?***
* Un `logit` es simplemente el valor crudo que sale de la última capa lineal de la red.
* Esto salia de `scores = model(xi)`.
* Si tienes 10 clases, la salida es algo como: `[ 2.3, -1.1, 0.7, 4.2, ... ]`
* Esos números:
  - No son probabilidades.
  - No están entre 0 y 1.
  - No suman 1.
  - Pueden ser negativos.
* ***¿Entonces qué es una predicción?***
* La predicción final es: `argmax(logits)`. ***Es decir, el índice del valor más alto***.
* `[2.3, -1.1, 0.7, 4.2] --> Clase 3`. Es un 3.

* ***¿Por qué en accuracy hacemos forward otra vez?***
* Durante entrenamiento haces: `forward → loss → backward → update`.
* Pero durante también hacemos: `accuracy(model, x_val, y_val)`.
* Estás evaluando en otro conjunto de datos (validación).
* Necesitamos:
  - Pasar esos datos por la red.
  - Obtener sus logits.
  - Ver qué clase predice.
  - Eso requiere otro forward.
* ***SOLO OCUPAMOS LA PREDICCION; NO ENTRENARLO EN ACCURACCY***
* "Después de aprender, ¿qué tan bien lo haces con datos nuevos?" Eso requiere otro forward..

* ***¿Qué es exactamente un epoch?***
* Un epoch = recorrer TODO el dataset una vez.
* Tienes: 100 datos y quieres minibatch size = 5
* Entonces: 100/5 = 20 minibatches por epoch.
* Si haces `epochs=50`.
* Entonces:
  - Cada uno de los 100 datos será usado 50 veces.
  - Cada minibatch será procesado 50 veces
$$20 \text{ batches} \times 50 \text{ epochs} = 1000 actualizaciones}$$

* ***CORRECTO***
* Si tienes 100 datos y mb_size = 5:
  - No tienes 5 minibatches.
  - Tienes 20 minibatches.
  - Cada uno de esos 20 minibatches se ejecutará 50 veces.


In [28]:
hidden1 = 1000
hidden = 1000
# Instacias del modelo.
# model1 = nn.Sequential(nn.Linear(in_features=784, out_features=hidden1),
#                        nn.ReLU(),
#                        nn.Linear(in_features=hidden1, out_features=hidden),
#                        nn.ReLU(),
#                        nn.Linear(in_features=hidden1, out_features=10)
#                       )
model1 = nn.Sequential(nn.Linear(in_features=784, out_features=hidden1),
                       nn.ReLU(),
                       nn.Linear(in_features=hidden1, out_features=10)
                      )


lr = 5e-2
epochs = 100
mb_size = 4096
optimiser = torch.optim.SGD(model1.parameters(), lr = lr)

train(model1, optimiser, mb_size, epochs)


Epoch: 0, costo: 1.110137939453125, accuracy: 0.83
Epoch: 1, costo: 0.6882022619247437, accuracy: 0.8693
Epoch: 2, costo: 0.536060631275177, accuracy: 0.8846
Epoch: 3, costo: 0.5142793655395508, accuracy: 0.8958
Epoch: 4, costo: 0.45593753457069397, accuracy: 0.8971
Epoch: 5, costo: 0.4549763798713684, accuracy: 0.9009
Epoch: 6, costo: 0.36484530568122864, accuracy: 0.9067
Epoch: 7, costo: 0.3760242462158203, accuracy: 0.9091
Epoch: 8, costo: 0.3636176884174347, accuracy: 0.9111
Epoch: 9, costo: 0.3609851002693176, accuracy: 0.9134
Epoch: 10, costo: 0.3330906331539154, accuracy: 0.9159
Epoch: 11, costo: 0.35060685873031616, accuracy: 0.9174
Epoch: 12, costo: 0.30443212389945984, accuracy: 0.9168
Epoch: 13, costo: 0.30260592699050903, accuracy: 0.9191
Epoch: 14, costo: 0.2808283865451813, accuracy: 0.9204
Epoch: 15, costo: 0.3453960120677948, accuracy: 0.9217
Epoch: 16, costo: 0.3135168254375458, accuracy: 0.9229
Epoch: 17, costo: 0.3093140721321106, accuracy: 0.925
Epoch: 18, costo: 0.

KeyboardInterrupt: 